© 2025 Amazon Web Services, Inc. or its affiliates. All Rights Reserved
This AWS Content is provided subject to the terms of the AWS Customer Agreement available at http://aws.amazon.com/agreement or other written agreement between Customer and either Amazon Web Services, Inc. or Amazon Web Services EMEA SARL or both.


# Asynchronous Data Analysis Agent 튜토리얼

## 개요

이 튜토리얼에서는 사용자와의 대화 응답성을 유지하면서 background에서 장시간 분석 task를 수행할 수 있는 asynchronous data analysis agent를 구축합니다. Amazon Bedrock AgentCore의 asynchronous 기능과 Strands를 활용하여 시간이 오래 걸리는 작업을 원활하게 처리하는 에이전트를 생성하는 방법을 보여줍니다.

### 사용 사례 세부 정보

| 항목                | 세부 정보                                                                    |
|---------------------|------------------------------------------------------------------------------|
| 사용 사례 유형      | Data Analysis Assistant                                                      |
| Agent 유형          | Asynchronous                                                                 |
| Agentic Framework   | Strands                                                                      |
| LLM 모델            | Anthropic Claude Sonnet 4(primary agent) 및 Haiku 4.5(coding agent)          |
| 구성 요소           | AgentCore Runtime, Async Tasks, Mock Analysis Tool                           |
| 예제 난이도         | 중급                                                                          |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK                                           |

### 사용 사례 아키텍처

이 data analysis assistant는 응답성이 뛰어난 AI 에이전트를 구축하는 강력한 pattern을 보여줍니다.

1. 사용자가 데이터 분석 요청
2. 에이전트가 AgentCore async 기능을 사용하여 background task 시작
3. background task가 실행되는 동안(~1 minute) 에이전트가 대화 계속
4. 결과가 준비되면 파일에 저장
5. 요청 시 에이전트가 결과를 가져와 제시

![아키텍처](architecture.png)

### Agent Flow 다이어그램

```
               사용자 요청
                    ↓
               Main Agent
                    ↓
               start_data_analysis_task()
                    ↓
              즉시 반환
              "Task started with ID: 12345"
                    ↓
         ┌──────────┴──────────┐
         │                     │
    Main Agent            Background Thread
    (응답 가능)            (async 처리)
         │                     │
    다른 질문에 응답            ↓
    계속                   1. S3에서 데이터 download
         │                     ↓
    "What is the capital   2. Start Code Interpreter
     of France?"               ↓
         │                 3. data.csv를 session에 쓰기
    output 반환                ↓
         │                 4. Coding Agent가 코드 생성
         │                    5-10s(LLM 호출)
         │                     ↓
         │                 5. Code Interpreter에서 코드 실행
         │                    15-30s
         │                     ↓
         │                 6. AgentCore와 S3에 결과 저장
         │                     ↓
         │                 7. task 완료 표시
         │                     │
         └─────────────────────┘
                    ↓
         User asks: "Show results"
                    ↓
               Main Agent
                    ↓
          get_task_status()
                    ↓
          get_task_result()
                    ↓
          완료된 결과 가져오기
                    ↓
          사용자에게 제시
```

#### 주요 아키텍처 결정: Coding Agent가 Async Task 내부에 있는 이유

Coding Agent는 사전 단계가 아니라 background task 내부에서 실행됩니다.

```
User → Main Agent → start_data_analysis_task() ← Returns in <1s
                         ↓
                    Background thread:
                      - Coding Agent가 코드 생성(5-10s)
                      - 코드 실행(소요 시간 가변)
```
이점: 사용자는 즉시 "task started" 응답을 받습니다. 비용이 큰 모든 작업(LLM 호출, 코드 실행)이 asynchronous 방식으로 수행됩니다. 전체 코드 생성 및 수정 과정이 완전히 asynchronous 방식으로 실행되어 main agent의 응답성을 최대한 유지합니다.

#### Agent Workflow 단계

1. **사용자 요청**: 사용자가 S3 URI와 함께 데이터 분석 query를 보냄
2. **Main Agent**: `start_data_analysis_task()` tool 호출
3. **즉시 반환**: main agent가 <1 second 이내에 task ID로 응답
4. **Background 처리**: 
   - S3에서 데이터 download
   - Code Interpreter session 시작
   - Coding Agent가 Python 코드 생성(여기서 LLM 호출)
   - 격리된 환경에서 코드 실행
   - AgentCore에 결과 저장
5. **병렬 대화**: main agent가 질문에 계속 응답
6. **Task 상태 확인**: main agent가 `get_task_status()`를 호출하여 task 상태 확인
6. **결과 가져오기**: main agent가 `get_analysis_tasks_info()`를 호출하여 완료된 task 결과 가져오기
7. **결과 제시**: main agent가 결과 형식을 지정하여 사용자에게 제시

### 주요 기능

* **Asynchronous Task Management**: 대화를 차단하지 않는 background task 시작
* **Code Interpreter 통합**: 격리된 안전한 환경에서 Python 코드 실행
* **S3 데이터 통합**: 자동 데이터 download 및 결과 저장
* **Embedded Coding Agent**: 응답성을 극대화하도록 LLM이 async task 내부에서 분석 코드 생성
* **Task 상태 Monitoring**: 활성 task 확인 및 완료된 결과 가져오기

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* 적절한 권한이 있는 AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands
* 실행 중인 Docker(로컬 테스트 및 배포용)


기술 설정을 살펴봤으므로 시작하기 전에 보안 고려 사항을 다루어야 합니다. 이 튜토리얼에는 자동화된 workflow에서 AI 생성 코드를 실행하는 과정이 포함되므로 악성 코드 injection을 방지하고 안전한 작동을 보장하려면 적절한 security guardrail을 구현해야 합니다.


## 보안: AI 코드 생성을 위한 Amazon Bedrock Guardrails

### 코드 보안이 중요한 이유

AI 에이전트가 코드를 자동으로 생성하고 실행할 때 **보안은 매우 중요합니다**. 적절한 보호 장치가 없으면 악성 prompt가 AI를 속여 다음과 같은 위험한 코드를 생성하게 할 수 있습니다.

- **system command 실행**(rm -rf /, sudo command)
- **민감한 데이터 액세스**(private 파일, environment variable 읽기)
- **network request 실행**(데이터 유출, malware download)
- **보안 제어 우회**(prompt injection 공격)

### Amazon Bedrock Guardrails Standard Tier

이 튜토리얼에서는 이러한 위험을 방지하도록 **code domain protection**을 적용한 **Amazon Bedrock Guardrails Standard Tier**를 구현합니다.

#### 🛡️ **Content Filters**
- **Misconduct**: malware, keylogger, security exploit 차단
- **Violence**: 물리적 피해를 일으킬 수 있는 코드 방지
- **Prompt Attacks**: jailbreak, injection 시도, prompt leakage 감지

#### 🔍 **Code Domain Intelligence**
- **12개 Programming Language**: Python, JavaScript, Java, C#, C++, PHP, Shell, HTML, SQL, C, GO, TypeScript
- **Context Awareness**: 코드와 텍스트, 정상 pattern과 악성 pattern 구분
- **PII Protection**: 코드 주석, variable, string의 민감한 데이터 감지

#### ⚡ **성능 이점**
- **Cross-Region Inference**: 높은 availability와 짧은 latency
- **Fail-Safe Design**: 실제 위협을 차단하면서 정상적인 분석 허용
- **AWS Native**: 전문적으로 유지 관리되며 자동 업데이트

### 보안 아키텍처

```
Flow: User Prompt → Bedrock Guardrails → AI Code Generation → Bedrock Guardrails → Code Execution

Example 1 (Legitimate Request): "Analyze data" → ✅ SAFE → Generates: "import pandas" → ✅ SAFE → ✓ Execute

Example 2 (Malicious Request): "rm -rf /" → ❌ BLOCKED → Would generate: "os.system('rm')" → ❌ BLOCKED → ✗ Prevent
```

이 **이중 검증**은 실행 전에 사용자 input과 AI 생성 코드가 모두 안전한지 확인합니다.

### Bedrock Guardrails 설정

다음 셀에서 이 튜토리얼에 사용할 Bedrock Guardrails를 자동으로 생성하고 구성합니다.

In [ ]:
# 필수 package 설치
!pip install -r requirements.txt --quiet

In [ ]:
# 코드 보안을 위한 Amazon Bedrock Guardrails Standard Tier 설정
import boto3
import time
import os


def setup_bedrock_guardrails():
    """코드 도메인 보호를 위해 Standard Tier의 Bedrock Guardrails를 생성하고 구성합니다."""

    bedrock_client = boto3.client("bedrock", region_name="us-east-2")

    guardrail_config = {
        "name": "async-data-analysis-code-safety",
        "description": "Code domain protection for async data analysis agent",
        "topicPolicyConfig": {
            "topicsConfig": [
                {
                    "name": "Malicious Code Generation",
                    "definition": "Code designed for malicious activities including malware, keyloggers, security exploits, system manipulation, or unauthorized access.",
                    "examples": [
                        "Generate keylogger code",
                        "Create malware script",
                        "Bypass security systems",
                    ],
                    "type": "DENY",
                }
            ]
        },
        "contentPolicyConfig": {
            "filtersConfig": [
                {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {
                    "type": "INSULTS",
                    "inputStrength": "MEDIUM",
                    "outputStrength": "MEDIUM",
                },
                {
                    "type": "MISCONDUCT",
                    "inputStrength": "HIGH",
                    "outputStrength": "HIGH",
                },
                {
                    "type": "PROMPT_ATTACK",
                    "inputStrength": "HIGH",
                    "outputStrength": "NONE",
                },
            ]
        },
        "sensitiveInformationPolicyConfig": {
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "BLOCK"},
                {"type": "PHONE", "action": "BLOCK"},
                {"type": "NAME", "action": "BLOCK"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
            ]
        },
        "blockedInputMessaging": "Request blocked by security guardrails for code safety.",
        "blockedOutputsMessaging": "Response blocked by security guardrails for code safety.",
        "tags": [{"key": "Purpose", "value": "AsyncDataAnalysis"}],
    }

    try:
        print("🛡️ Creating Bedrock Guardrails with Standard Tier...")
        response = bedrock_client.create_guardrail(**guardrail_config)

        guardrail_id = response["guardrailId"]
        print(f"✅ Guardrail created: {guardrail_id}")

        # guardrail이 준비될 때까지 대기
        while True:
            status_response = bedrock_client.get_guardrail(guardrailIdentifier=guardrail_id)
            if status_response["status"] == "READY":
                break
            time.sleep(5)

        # 에이전트용 environment variable 설정
        os.environ["BEDROCK_GUARDRAIL_ID"] = guardrail_id

        print(f"🔒 Bedrock Guardrails ready with ID: {guardrail_id}")
        print("✅ Code security protection enabled")

        return guardrail_id

    except Exception as e:
        if "already exists" in str(e) or "ConflictException" in str(e):
            # guardrail이 이미 존재하므로 기존 항목 가져오기
            guardrails = bedrock_client.list_guardrails()
            for guardrail in guardrails["guardrails"]:
                if guardrail["name"] == "async-data-analysis-code-safety":
                    guardrail_id = guardrail["id"]
                    os.environ["BEDROCK_GUARDRAIL_ID"] = guardrail_id
                    print(f"✅ Using existing guardrail: {guardrail_id}")
                    return guardrail_id

        print(f"❌ Error setting up guardrails: {e}")
        print("⚠️ Continuing without guardrails - security features will be limited")
        return None


# guardrail 자동 설정
guardrail_id = setup_bedrock_guardrails()
if guardrail_id:
    print(f"\n🎯 Environment variable set: BEDROCK_GUARDRAIL_ID={guardrail_id}")
    print("🔐 All AI-generated code will be validated for security before execution")

## 코드 이해

asynchronous data analysis agent 구현의 주요 구성 요소를 살펴봅니다.

**주요 구성 요소:**

1. **AgentCore 통합**: async task management를 위한 BedrockAgentCoreApp 초기화
2. **Async Task Tool**: `async_analysis_task()`가 coding agent와 함께 background thread 시작
3. **Code Interpreter**: 격리된 session에서 생성된 Python 코드를 실행하고 오류 발생 시 retry

In [ ]:
# syntax highlighting으로 entrypoint 코드 표시
# !pygmentize async_data_analysis_agent.py

### Agent Architecture 이해

**Main Agent(Orchestrator):**
  - 🔧 Tool 1: `async_analysis_task` - background task 시작
  - 🔧 Tool 2: `get_task_status` - task 진행 상황 확인  
  - 🔧 Tool 3: `get_task_results` - 완료된 결과 가져오기
  - 📞 호출: Amazon Bedrock LLM(Claude Sonnet 4)

  **Coding Agent(Background):**
  - 🔧 Tools: 없음
  - 💻 목적: 사용자 요청에 따라 Python 코드 생성
  - 📞 호출: Amazon Bedrock LLM(Claude Haiku 4.5)
  - 🏃 실행: async background task 내부에서만 실행

### Runtime Role 생성

에이전트를 배포하기 전에 이 advanced 튜토리얼에 필요한 **특별 권한**이 있는 IAM role을 생성해야 합니다. 

**Role을 수동으로 생성하는 이유**

이 asynchronous data analysis agent에는 기본 agent 튜토리얼과 달리 표준 AgentCore role template에 포함되지 않은 추가 AWS 권한이 필요합니다.

- 🔧 **Code Interpreter**: 격리된 session에서 Python 코드 실행
- 📦 **S3 액세스**: input data download 및 분석 결과 업로드
- 🛡️ **Bedrock Guardrails**: 보안을 위해 AI 생성 코드 검증

`utils.py`의 `create_agentcore_role()` 함수로 기본 role을 생성하고 `add_permissions()`를 사용하여 이러한 추가 기능을 부여합니다.

In [ ]:
import json
from utils import create_agentcore_role


# 권한 추가
def add_permissions(role_name):
    """
    기존 IAM 역할에 Code Interpreter 및 S3 권한을 추가합니다.

    매개변수:
        role_name: 업데이트할 IAM 역할 이름

    반환값:
        put_role_policy 응답
    """
    import boto3

    iam_client = boto3.client("iam")

    # 특정 리소스 ARN에 사용할 account ID와 region 가져오기
    boto3.client("sts").get_caller_identity()["Account"]
    boto3.Session().region_name or "us-east-2"

    policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "CodeInterpreterPermissions",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:StartCodeInterpreterSession",
                    "bedrock-agentcore:StopCodeInterpreterSession",
                    "bedrock-agentcore:SendCodeInterpreterMessage",
                    "bedrock-agentcore:GetCodeInterpreterSession",
                    "bedrock-agentcore:InvokeCodeInterpreter",
                    "bedrock-agentcore:ListCodeInterpreterSessions",
                    "bedrock-agentcore:*",
                ],
                "Resource": "*",
            },
            {
                "Sid": "S3ReadWritePermissions",
                "Effect": "Allow",
                "Action": [
                    "s3:GetObject",
                    "s3:PutObject",
                    "s3:DeleteObject",
                    "s3:ListBucket",
                    "s3:GetBucketLocation",
                    "s3:ListAllMyBuckets",
                ],
                "Resource": "*",
            },
            {
                "Sid": "BedrockModelAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream",
                    "bedrock:ApplyGuardrail",
                    "bedrock:GetGuardrail",
                    "bedrock:ListGuardrails",
                    "bedrock:CreateGuardrail",
                    "bedrock:UpdateGuardrail",
                    "bedrock:CreateGuardrailVersion",
                    "bedrock:GetGuardrailVersion",
                ],
                "Resource": "*",
            },
            {
                "Sid": "CloudWatchLogsAccess",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                ],
                "Resource": "*",
            },
        ],
    }

    try:
        response = iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName="AsyncDataAnalysisPolicy",
            PolicyDocument=json.dumps(policy),
        )
        print(f"Code Interpreter and S3 permissions added to role: {role_name}")
        return response
    except Exception as e:
        print(f"Error adding Code Interpreter and S3 permissions: {e}")
        raise


agent_name = "data_analysis_agent"
agentcore_iam_role = create_agentcore_role(agent_name=agent_name)

# 권한 추가
add_permissions(agentcore_iam_role["Role"]["RoleName"])

### AgentCore Runtime 배포 구성

이제 AgentCore starter toolkit을 사용하여 에이전트 배포를 구성합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="async_data_analysis_agent.py",
    execution_role=agentcore_iam_role["Role"]["Arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
)
response

### Synthetic Data 생성

분석에 사용할 synthetic time series data를 생성합니다. 이 데이터는 S3 bucket에 저장되어 에이전트 테스트에 사용됩니다. 데이터는 6-year period(2010-2015)에 걸친 5개 제품의 월별 가격을 시뮬레이션합니다: (1) Phone, (2) TV, (3) Shoes, (4) Refrigerator, (5) Laptop.

synthetic 가격에는 점진적인 상승 추세, 계절 변동(겨울에는 높고 여름에는 낮음), random noise가 포함됩니다. 가격은 각 제품 base price의 70% 이상으로 제한됩니다.

In [ ]:
# 간단한 time series data 생성
import pandas as pd
import numpy as np
from datetime import datetime


def create_simple_time_series_data(seed=67):
    # 재현성을 위한 seed 설정
    np.random.seed(seed)
    products = ["Phone", "TV", "Shoes", "Refrigerator", "Laptop"]
    dates = pd.date_range(start="2010-01-01", end="2015-12-31", freq="M")
    data = []
    for product in products:
        # 각 제품의 간단한 base price
        base_prices = {
            "Phone": 800,
            "TV": 1200,
            "Shoes": 120,
            "Refrigerator": 600,
            "Laptop": 1200,
        }
        base_price = base_prices[product]
        for i, date in enumerate(dates):
            # 간단한 추세(시간에 따른 소폭 가격 상승)
            trend = i * 0.05
            # 간단한 계절성(겨울에는 가격이 높고 여름에는 낮음)
            seasonal = 20 * np.sin(2 * np.pi * (date.dayofyear - 80) / 365)
            # 무작위 noise
            noise = np.random.normal(0, 10)
            # 최종 가격
            price = base_price + trend + seasonal + noise
            price = max(price, base_price * 0.7)  # base price의 70% 미만으로 내리지 않음
            data.append(
                {
                    "date": date.strftime("%Y-%m-%d"),
                    "product": product,
                    "price": round(price, 2),
                }
            )
    return pd.DataFrame(data)


# 데이터 생성
sample_df = create_simple_time_series_data()
sample_df["date"] = pd.to_datetime(sample_df["date"])
print(f"Created {len(sample_df)} records for {len(sample_df['product'].unique())} products")
print(f"Date range: {sample_df['date'].min()} to {sample_df['date'].max()}")
print("\nSample data:")
print(sample_df.head(10))

In [ ]:
sample_df.loc[sample_df["product"] == "Laptop"].plot(x="date", y="price", title="Laptop Price")

### S3 Bucket에 데이터 저장

데이터를 저장할 S3 bucket을 생성합니다. 에이전트는 S3 bucket에 액세스하여 데이터를 Code Interpreter로 전달할 수 있습니다.

In [ ]:
# S3 bucket 생성

import io
import uuid
import pandas as pd
from botocore.exceptions import ClientError


def create_s3_bucket(bucket_name, region=None):
    """지정한 리전에 S3 버킷을 생성합니다."""
    try:
        # 지정되지 않은 경우 현재 session region 가져오기
        if region is None:
            session = boto3.Session()
            region = session.region_name or "us-west-2"

        print(f"Using region: {region}")
        s3_client = boto3.client("s3", region_name=region)

        if region == "us-east-1":
            # us-east-1에는 LocationConstraint가 필요하지 않음
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )

        print(f"Bucket '{bucket_name}' created successfully in {region}")
        return True, region
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "BucketAlreadyOwnedByYou":
            print(f"Bucket '{bucket_name}' already exists and is owned by you")
            return True, region
        elif error_code == "BucketAlreadyExists":
            print(f"Bucket name '{bucket_name}' already exists globally")
            return False, region
        else:
            print(f"Error creating bucket: {e}")
            return False, region


sts = boto3.client("sts")
identity = sts.get_caller_identity()
# 고유한 bucket 이름 생성
account_id = identity.get("Account", "unknown") if "identity" in globals() else "unknown"
unique_id = str(uuid.uuid4())[:8]
bucket_name = f"my-data-bucket-{account_id}-{unique_id}"

print(f"Creating bucket: {bucket_name}")
bucket_created, bucket_region = create_s3_bucket(bucket_name)

In [ ]:
# S3에 데이터 업로드
file_key = f"data/sample_products_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# DataFrame을 CSV string으로 변환
csv_buffer = io.StringIO()
sample_df.to_csv(csv_buffer, index=False)

s3_client = boto3.client("s3")

# S3에 업로드
s3_client.put_object(Bucket=bucket_name, Key=file_key, Body=csv_buffer.getvalue(), ContentType="text/csv")

# S3 URI 가져오기
s3_uri = f"s3://{bucket_name}/{file_key}"
print(f"Uploaded to: {s3_uri}")

### AgentCore Runtime에 Agent 시작

에이전트 구성을 마쳤으므로 AgentCore Runtime에 시작합니다.

In [ ]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
launch_result

### AgentCore Runtime 상태 확인

에이전트의 배포 상태를 확인합니다. 상태가 "READY"가 될 때까지 기다린 후 계속 진행합니다.

In [ ]:
status_response = agentcore_runtime.status()
print(f"Agent Status: {status_response.endpoint['status']}")

In [ ]:
def display_agent_response(invoke_response):
    """JSON 파싱과 Markdown 렌더링을 적용해 에이전트 응답을 표시하는 유틸리티 함수입니다."""
    from IPython.display import Markdown, display
    import json

    try:
        full_response = "".join(invoke_response["response"])
        parsed = json.loads(full_response)
        text = parsed["result"]["content"][0]["text"]
        display(Markdown(text))
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"Error parsing response: {e}")
        print(f"Raw response: {full_response if 'full_response' in locals() else invoke_response}")

### Asynchronous Data Analysis Agent 작동 확인

데이터 분석 요청으로 에이전트를 호출하면 대화를 계속하면서 background task를 시작합니다.

에이전트는 다음 작업을 수행합니다.
* 요청 확인
* background analysis task 시작
* 계속 질문할 수 있도록 지원

AWS Console의 Amazon Bedrock AgentCore에서 task 상태를 monitoring할 수 있습니다.

In [ ]:
# async analysis가 필요한 질문 제출
start_time = time.time()
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": f"Load data from {s3_uri} and calculate the average price by product. Show the top 3 products by price in a formatted table."
    }
)
print(f"Response time: {time.time() - start_time:.2f}s")
display_agent_response(invoke_response)
print("=" * 80)

In [ ]:
# async 실행 중 질문 제출
start_time = time.time()
invoke_response = agentcore_runtime.invoke(
    {"prompt": "While that runs, explain what 'average price by product' means in data analysis terms."}
)
print(f"Response time: {time.time() - start_time:.2f}s")
display_agent_response(invoke_response)
print("=" * 80)

In [ ]:
# task 상태 확인
start_time = time.time()
invoke_response = agentcore_runtime.invoke({"prompt": "What is the status of the data analysis task?"})
print(f"Response time: {time.time() - start_time:.2f}s")
display_agent_response(invoke_response)

In [ ]:
# 30 seconds 동안 실행 일시 중지
time.sleep(30)

# task 상태 확인
start_time = time.time()
invoke_response = agentcore_runtime.invoke({"prompt": "Get the results for that task and show me the findings."})
print(f"Response time: {time.time() - start_time:.2f}s")
display_agent_response(invoke_response)

## 마무리

이 튜토리얼에서는 Strands와 Amazon Bedrock AgentCore를 사용하여 asynchronous data analysis agent를 구축했습니다. 에이전트는 다음 작업을 수행할 수 있습니다.

1. 사용자의 데이터 분석 요청 수신
2. 분석을 수행할 background task 시작
3. 분석이 실행되는 동안 사용자와 대화 계속
4. 완료 시 분석 결과를 파일에 저장
5. 사용자 요청 시 결과를 가져와 제시

이 pattern은 task 완료에 시간이 걸리지만 사용자가 빠른 상호 작용을 기대하는 애플리케이션에 유용합니다.

### 핵심 요점

- **Asynchronous agent**는 시간이 오래 걸리는 task에 더 나은 사용자 경험 제공
- **AgentCore Runtime**으로 에이전트를 쉽게 배포하고 확장
- **Strands**가 유연한 multi-agent orchestration 제공
- **File System 통합**을 통해 에이전트가 결과를 저장하고 가져올 수 있음

## 리소스 정리(선택 사항)

이 튜토리얼에서 생성한 리소스를 정리하려면 다음 코드를 사용합니다.

In [ ]:
# agent ID와 ECR URI 가져오기
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# client 초기화
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)
iam_client = boto3.client("iam")

# agent runtime 삭제
runtime_delete_response = agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)

# ECR repository 삭제
response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# IAM role 삭제
policies = iam_client.list_role_policies(RoleName=agentcore_iam_role["Role"]["RoleName"], MaxItems=100)

for policy_name in policies["PolicyNames"]:
    iam_client.delete_role_policy(RoleName=agentcore_iam_role["Role"]["RoleName"], PolicyName=policy_name)
iam_response = iam_client.delete_role(RoleName=agentcore_iam_role["Role"]["RoleName"])